In [ ]:
import pandas as pd
import wandb
import matplotlib.pyplot as plt
from sortedcontainers import SortedDict


wandb.login()

api = wandb.Api()
project_name = "WB_CLTV"
runs = api.runs(path=project_name)

# Extract data
def extract_data(runs):
    data = []
    for run in runs:
        config = dict(run.config)
        runtime = run.summary.get("_runtime", None)
        seed = config.get("seed", None)
        env = config.get("env", None)
        dataset_types = config.get("dataset_types", None)

        if all([runtime, seed]): 
            data_point = {
                'env': env,
                'method': config.get('method', None),
                'dataset_types': dataset_types,
                'baseline': config.get('baseline', None),
                'seed': seed,
                '_runtime': runtime
            }
            data.append(data_point)

    return pd.DataFrame(data)

df = extract_data(runs)


df['_runtime'] /= 3600  

In [ ]:
df.head()

In [ ]:
df.dataset_types = df.dataset_types.astype("str")
df.rename(columns={'_runtime': 'runtime'}, inplace=True)


datasets_t = {
    "['random-v2', 'expert-v2']": "random-expert", 
    "['random-v2', 'medium-v2']": "random-medium", 
    "['medium-v2', 'expert-v2']": "medium-expert"
}
df['dataset_types'] = df['dataset_types'].map(datasets_t)

domains = ["Ant", "HalfCheetah", "Hopper", "Walker2d"]
datasets = {dataset: ["random-medium", "random-expert", "medium-expert"] for dataset in domains}


In [ ]:
fig = plt.figure(figsize=(15, 15))
plt.subplots_adjust(hspace=0.3)
axes = fig.subplots(nrows=4, ncols=3)

width = 0.8  
methods = ["Vanilla", "CUORL", "Harness", "CLTV"] 
base_models = ["CQL", "IQL"]
color_map = {'Vanilla': "coral", 'Harness': "dodgerblue", 'CUORL': "crimson", 'CLTV': "green"}

for row, (domain, dataset_names) in enumerate(datasets.items()):
    for col, dataset_name in enumerate(dataset_names):
        subset_df = df[(df['env'] == domain) & (df['dataset_types'] == dataset_name)]
        if not subset_df.empty:
            unique_tasks = sorted(subset_df['method'].unique(), key=lambda x: methods.index(x) if x in methods else len(methods))

            for j, baseline in enumerate(base_models):
                for i, task in enumerate(unique_tasks):
                    task_baseline_df = subset_df[(subset_df['method'] == task) & (subset_df['baseline'] == baseline)]
                    mean_runtime = task_baseline_df['runtime'].mean()
                    std_runtime = task_baseline_df['runtime'].std()
                    position = j * (len(unique_tasks) + 1) + i
                    color = color_map.get(task, "default_color")
                    
                    axes[row, col].bar(position, mean_runtime, yerr=std_runtime,
                                       color=color, width=width, label=task)

            axes[row, col].set_title(dataset_name, fontsize=18, color="darkslategray")
            axes[row, col].set_xticks([j * (len(unique_tasks) + 1) + 0.5 * len(unique_tasks) for j in range(len(base_models))])
            axes[row, col].set_xticklabels(base_models, fontsize=16)
            axes[row, col].tick_params(axis='x', labelsize=15)

        if col == 0: 
            axes[row, col].text(-0.30, 0.5, domain.capitalize(), color="midnightblue", transform=axes[row, col].transAxes, fontsize=20, va='center', rotation='vertical')
            axes[row, col].set_ylabel('Runtime (hours)', fontsize=16)
        else:
            axes[row, col].set_ylabel('')

        for axl in axes.flatten():
            axl.tick_params(axis='x', labelsize=14)
            axl.tick_params(axis='y', labelsize=14)

handles, labels = axes[0, 0].get_legend_handles_labels()
unique_labels = ["Vanilla", "CUORL", "Harness", "CLTV"]
unique_handles = [handles[labels.index(label)] for label in unique_labels]

plt.tight_layout()
fig.legend(unique_handles, unique_labels, loc='upper center', ncol=4, fontsize=18, bbox_to_anchor=(0.5, 1.05))
plt.show()